In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 3.4.1


In [3]:
data_path = './data/yellow_tripdata_2025-11.parquet'

In [4]:
df = spark.read.parquet(data_path)

In [5]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [6]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [7]:
# Repartition into 4 partitions
df = df.repartition(4)

In [8]:
# Write back (folder with 4 parquet files)
df.write.mode("overwrite").parquet("data/partitions/yellow_tripdata_2025-11")

### Q3

In [9]:
from pyspark.sql.functions import to_date

In [10]:
df_filtered = df.filter(to_date("tpep_pickup_datetime") == "2025-11-15")
trip_count = df_filtered.count()
print(f"Number of trips on 2025-11-15: {trip_count}")

Number of trips on 2025-11-15: 162604


### Q4

In [11]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [12]:
from pyspark.sql.functions import unix_timestamp

In [13]:
# Calculate trip_duration in minutes
df = df.withColumn("trip_duration", (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60)

In [14]:
# Verify
df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration").show(5)

+--------------------+---------------------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|    trip_duration|
+--------------------+---------------------+-----------------+
| 2025-11-03 19:01:48|  2025-11-03 19:08:48|              7.0|
| 2025-11-10 12:46:18|  2025-11-10 13:03:42|             17.4|
| 2025-11-02 14:17:02|  2025-11-02 14:25:34|8.533333333333333|
| 2025-11-08 20:05:28|  2025-11-08 20:13:12|7.733333333333333|
| 2025-11-02 15:30:08|  2025-11-02 15:39:37|9.483333333333333|
+--------------------+---------------------+-----------------+
only showing top 5 rows



In [15]:
from pyspark.sql.functions import max

In [16]:
df.select(max("trip_duration")/60).show()

+-------------------------+
|(max(trip_duration) / 60)|
+-------------------------+
|        90.64666666666668|
+-------------------------+



### Q6

In [17]:
data_path = './data/taxi_zone_lookup.csv'

In [18]:
# Read the CSV into a DataFrame
df_zones = (
    spark.read
    .option("header", "true")     # CSV has header
    .option("inferSchema", "true")  # Optional but useful
    .csv(data_path)
)

# Create a temporary view
df_zones.createOrReplaceTempView("taxi_zones")

In [19]:
# Verify
spark.sql("SELECT * FROM taxi_zones LIMIT 5").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+



In [30]:
# Join for pickup zone
df_merged = df.join(df_zones.select("LocationID", "Zone").alias("pu_zones"), df.PULocationID == df_zones.LocationID, "left") \
            .withColumnRenamed("Zone", "PUZone") \
            .drop("LocationID")

In [31]:
# Verify
df_merged.select("PULocationID", "PUZone").show(5)

+------------+---------+
|PULocationID|   PUZone|
+------------+---------+
|         196|Rego Park|
|         196|Rego Park|
|         196|Rego Park|
|         196|Rego Park|
|         196|Rego Park|
+------------+---------+
only showing top 5 rows



In [ ]:
df_merged.groupBy("PUZone").count().sort("count", ascending=True).show(5)